In [1]:
import pandas as pd

nav_df = pd.read_csv("data/raw/02_nav_history.csv")

nav_df.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [2]:
# Convert date column to datetime
nav_df['date'] = pd.to_datetime(nav_df['date'])

# data types
print(nav_df.dtypes)

# Missing values
print("\nMissing Values:")
print(nav_df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:", nav_df.duplicated().sum())

# NAV <= 0
print("\nInvalid NAV Records:", (nav_df['nav'] <= 0).sum())

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object

Missing Values:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate Rows: 0

Invalid NAV Records: 0


In [3]:
# Sort by fund and date
nav_df = nav_df.sort_values(
    by=['amfi_code', 'date']
).reset_index(drop=True)

nav_df.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [4]:
# Forward fill NAV within each fund
nav_df['nav'] = (
    nav_df.groupby('amfi_code')['nav']
    .ffill()
)

In [5]:
print(nav_df.isnull().sum())

amfi_code    0
date         0
nav          0
dtype: int64


In [6]:
import os

os.makedirs("data/processed", exist_ok=True)

In [7]:
nav_df.to_csv(
    "data/processed/nav_history_clean.csv",
    index=False
)

print("Cleaned NAV dataset saved successfully!")

Cleaned NAV dataset saved successfully!


In [8]:
txn_df = pd.read_csv("data/raw/08_investor_transactions.csv")
txn_df.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [9]:
# data types
txn_df.info()

# Missing values
print("\nMissing Values:")
print(txn_df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:", txn_df.duplicated().sum())

# Unique transaction types
print("\nTransaction Types:")
print(txn_df['transaction_type'].unique())

# Unique KYC Status
print("\nKYC Status:")
print(txn_df['kyc_status'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
tran

In [10]:
txn_df['transaction_type'] = (
    txn_df['transaction_type']
    .str.strip()
    .str.title()
)

# Validate allowed values
allowed_txn = ['Sip', 'Lumpsum', 'Redemption']

invalid_txn = txn_df[
    ~txn_df['transaction_type'].isin(allowed_txn)
]

print("Invalid Transaction Types:")
print(invalid_txn[['transaction_type']].drop_duplicates())

Invalid Transaction Types:
Empty DataFrame
Columns: [transaction_type]
Index: []


In [11]:
txn_df['transaction_date'] = pd.to_datetime(
    txn_df['transaction_date']
)

print(txn_df['transaction_date'].dtype)

datetime64[ns]


In [12]:
invalid_amounts = txn_df[
    txn_df['amount_inr'] <= 0
]

print("Invalid Amount Records:", len(invalid_amounts))

Invalid Amount Records: 0


In [13]:
allowed_kyc = [
    'Verified',
    'Pending',
    'Rejected'
]

invalid_kyc = txn_df[
    ~txn_df['kyc_status'].isin(allowed_kyc)
]

print("Invalid KYC Records:")
print(invalid_kyc[['kyc_status']].drop_duplicates())

Invalid KYC Records:
Empty DataFrame
Columns: [kyc_status]
Index: []


In [14]:
perf_df = pd.read_csv("data/raw/07_scheme_performance.csv")

perf_df.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [15]:
# Dataset info
perf_df.info()

# Missing values
print("\nMissing Values:")
print(perf_df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:", perf_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [16]:
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'benchmark_3yr_pct'
]

for col in return_cols:
    perf_df[col] = pd.to_numeric(
        perf_df[col],
        errors='coerce'
    )

print(perf_df[return_cols].isnull().sum())

return_1yr_pct       0
return_3yr_pct       0
return_5yr_pct       0
benchmark_3yr_pct    0
dtype: int64


In [17]:
invalid_expense = perf_df[
    (perf_df['expense_ratio_pct'] < 0.1) |
    (perf_df['expense_ratio_pct'] > 2.5)
]

print("Invalid Expense Ratio Records:")
print(len(invalid_expense))

Invalid Expense Ratio Records:
0


In [18]:
perf_df.to_csv(
    "data/processed/07_scheme_performance_clean.csv",
    index=False
)

print("Scheme Performance cleaned and saved successfully!")

Scheme Performance cleaned and saved successfully!


In [19]:
schema_sql = """
CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    scheme_name TEXT,
    fund_house TEXT,
    category TEXT,
    plan TEXT,
    risk_grade TEXT
);

CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE,
    year INTEGER,
    quarter INTEGER,
    month INTEGER,
    month_name TEXT
);

CREATE TABLE fact_nav (
    amfi_code INTEGER,
    date DATE,
    nav REAL
);

CREATE TABLE fact_transactions (
    investor_id TEXT,
    transaction_date DATE,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT
);

CREATE TABLE fact_performance (
    amfi_code INTEGER,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL
);

CREATE TABLE fact_aum (
    amfi_code INTEGER,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER
);
"""

with open("sql/schema.sql", "w") as f:
    f.write(schema_sql)

print("All fact tables added successfully!")

All fact tables added successfully!


In [20]:
import sqlite3

# Create/connect database file
conn = sqlite3.connect("bluestock_mf.db")

print("SQLite database created successfully!")
conn.close()

SQLite database created successfully!


In [21]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")
cursor = conn.cursor()

In [23]:
txn_df.to_csv(
    "data/processed/investor_transactions_clean.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [25]:
import os
print(os.listdir("data/processed"))

['07_scheme_performance_clean.csv', 'investor_transactions_clean.csv', 'nav_history_clean.csv']


In [27]:
import pandas as pd

nav_df = pd.read_csv("data/processed/nav_history_clean.csv")
txn_df = pd.read_csv("data/processed/investor_transactions_clean.csv")
perf_df = pd.read_csv("data/processed/07_scheme_performance_clean.csv")

In [29]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")
cursor = conn.cursor()

print("Database created and connected successfully!")

Database created and connected successfully!


In [31]:
import pandas as pd

nav_df = pd.read_csv("data/processed/nav_history_clean.csv")
txn_df = pd.read_csv("data/processed/investor_transactions_clean.csv")
perf_df = pd.read_csv("data/processed/07_scheme_performance_clean.csv")

In [33]:
nav_df.to_sql("fact_nav", conn, if_exists="replace", index=False)
txn_df.to_sql("fact_transactions", conn, if_exists="replace", index=False)
perf_df.to_sql("fact_performance", conn, if_exists="replace", index=False)

print("All tables loaded into SQLite!")

All tables loaded into SQLite!


In [35]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)

print(tables)

                name
0           fact_nav
1  fact_transactions
2   fact_performance


In [37]:
pd.read_sql("SELECT * FROM fact_nav LIMIT 5", conn)

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [39]:
pd.read_sql("""
SELECT amfi_code, MAX(date) as latest_date, nav
FROM fact_nav
GROUP BY amfi_code
ORDER BY nav DESC
LIMIT 5;
""", conn)

,amfi_code,latest_date,nav
0,120844,2026-05-29,4268.5497
1,125497,2026-05-29,1204.9571
2,101206,2026-05-29,773.2939
3,149322,2026-05-29,606.2349
4,100016,2026-05-29,583.6113


In [41]:
pd.read_sql("""
SELECT amfi_code, AVG(nav) as avg_nav
FROM fact_nav
GROUP BY amfi_code
ORDER BY avg_nav DESC
LIMIT 5;
""", conn)

,amfi_code,avg_nav
0,120844,3705.876331
1,125497,797.454477
2,100016,566.191221
3,149322,439.176645
4,101206,435.975472


In [43]:
pd.read_sql("""
SELECT transaction_type, COUNT(*) as total_transactions
FROM fact_transactions
GROUP BY transaction_type;
""", conn)

,transaction_type,total_transactions
0,Lumpsum,8095
1,Redemption,4967
2,Sip,19716


In [45]:
pd.read_sql("""
SELECT state, SUM(amount_inr) as total_investment
FROM fact_transactions
GROUP BY state
ORDER BY total_investment DESC
LIMIT 5;
""", conn)

,state,total_investment
0,Punjab,315780459
1,Tamil Nadu,315177237
2,Madhya Pradesh,308312493
3,Rajasthan,298645822
4,Gujarat,298358940


In [47]:
pd.read_sql("""
SELECT transaction_type, SUM(amount_inr) as total_amount
FROM fact_transactions
GROUP BY transaction_type;
""", conn)

,transaction_type,total_amount
0,Lumpsum,2059821448
1,Redemption,1244525491
2,Sip,217233491


In [49]:
pd.read_sql("""
SELECT amfi_code, COUNT(*) as nav_records
FROM fact_nav
GROUP BY amfi_code
ORDER BY nav_records DESC
LIMIT 5;
""", conn)

,amfi_code,nav_records
0,149324,1150
1,149323,1150
2,149322,1150
3,148569,1150
4,148568,1150


In [51]:
pd.read_sql("""
SELECT strftime('%Y-%m', transaction_date) as month,
       SUM(amount_inr) as sip_total
FROM fact_transactions
WHERE transaction_type = 'Sip'
GROUP BY month
ORDER BY month;
""", conn)

,month,sip_total
0,2024-01,12635349
1,2024-02,12613376
2,2024-03,12088413
3,2024-04,13512385
4,2024-05,13218606
5,2024-06,13131150
6,2024-07,13513884
7,2024-08,12521433
8,2024-09,12288778
9,2024-10,12467875


In [53]:
pd.read_sql("""
SELECT transaction_type,
       AVG(amount_inr) as avg_amount
FROM fact_transactions
GROUP BY transaction_type;
""", conn)

,transaction_type,avg_amount
0,Lumpsum,254456.015812
1,Redemption,250558.786189
2,Sip,11018.132025


In [55]:
pd.read_sql("""
SELECT city,
       SUM(amount_inr) as total_investment
FROM fact_transactions
GROUP BY city
ORDER BY total_investment DESC
LIMIT 5;
""", conn)

,city,total_investment
0,Kolkata,297182514
1,Hyderabad,290219284
2,Gurugram,223778908
3,Coimbatore,170345165
4,Bhopal,165606495


In [57]:
pd.read_sql("""
SELECT investor_id,
       SUM(amount_inr) as total_invested
FROM fact_transactions
GROUP BY investor_id
ORDER BY total_invested DESC
LIMIT 5;
""", conn)

,investor_id,total_invested
0,INV003288,3480685
1,INV001999,3179916
2,INV004605,2882975
3,INV000477,2846388
4,INV001692,2751086


In [59]:
queries_sql = """
-- 1. Top NAV funds
SELECT amfi_code, MAX(date), nav FROM fact_nav GROUP BY amfi_code ORDER BY nav DESC LIMIT 5;

-- 2. Average NAV
SELECT amfi_code, AVG(nav) FROM fact_nav GROUP BY amfi_code ORDER BY AVG(nav) DESC LIMIT 5;

-- 3. Transaction counts
SELECT transaction_type, COUNT(*) FROM fact_transactions GROUP BY transaction_type;

-- 4. State investment
SELECT state, SUM(amount_inr) FROM fact_transactions GROUP BY state ORDER BY SUM(amount_inr) DESC LIMIT 5;

-- 5. SIP vs Lumpsum
SELECT transaction_type, SUM(amount_inr) FROM fact_transactions GROUP BY transaction_type;

-- 6. Active funds
SELECT amfi_code, COUNT(*) FROM fact_nav GROUP BY amfi_code ORDER BY COUNT(*) DESC LIMIT 5;

-- 7. SIP trend
SELECT strftime('%Y-%m', transaction_date), SUM(amount_inr) FROM fact_transactions WHERE transaction_type='Sip' GROUP BY 1;

-- 8. Avg transaction amount
SELECT transaction_type, AVG(amount_inr) FROM fact_transactions GROUP BY transaction_type;

-- 9. City investment
SELECT city, SUM(amount_inr) FROM fact_transactions GROUP BY city ORDER BY SUM(amount_inr) DESC LIMIT 5;

-- 10. Top investors
SELECT investor_id, SUM(amount_inr) FROM fact_transactions GROUP BY investor_id ORDER BY SUM(amount_inr) DESC LIMIT 5;
"""

with open("sql/queries.sql", "w") as f:
    f.write(queries_sql)

print("queries.sql created successfully!")

queries.sql created successfully!


In [63]:
import os

os.makedirs("docs", exist_ok=True)

print("docs folder ready")

docs folder ready


In [65]:
with open("docs/data_dictionary.md", "w", encoding="utf-8") as f:
    f.write(data_dictionary)

print("Data dictionary created successfully!")

Data dictionary created successfully!


In [67]:
import os
print(os.listdir("docs"))

['data_dictionary.md']


In [69]:
data_dictionary = """
# 📘 Data Dictionary – Mutual Fund Analytics Project

## 1. nav_history_clean

| Column | Data Type | Description |
|--------|----------|-------------|
| amfi_code | Integer | Unique identifier for mutual fund scheme |
| date | Date | NAV date |
| nav | Float | Net Asset Value of the scheme |

---

## 2. investor_transactions_clean

| Column | Data Type | Description |
|--------|----------|-------------|
| investor_id | Text | Unique investor ID |
| transaction_date | Date | Date of transaction |
| amfi_code | Integer | Scheme identifier |
| transaction_type | Text | Type of transaction (SIP, Lumpsum, Redemption) |
| amount_inr | Float | Transaction amount in INR |
| state | Text | Investor state |
| city | Text | Investor city |
| city_tier | Text | City classification (T30/B30) |
| age_group | Text | Age category of investor |
| gender | Text | Gender of investor |
| annual_income_lakh | Float | Annual income in lakhs |
| payment_mode | Text | Mode of payment |
| kyc_status | Text | KYC verification status |

---

## 3. scheme_performance_clean

| Column | Data Type | Description |
|--------|----------|-------------|
| amfi_code | Integer | Scheme identifier |
| scheme_name | Text | Name of mutual fund scheme |
| fund_house | Text | Asset management company |
| category | Text | Fund category |
| plan | Text | Direct / Regular plan |
| return_1yr_pct | Float | 1-year return percentage |
| return_3yr_pct | Float | 3-year return percentage |
| return_5yr_pct | Float | 5-year return percentage |
| benchmark_3yr_pct | Float | Benchmark return |
| alpha | Float | Excess return over benchmark |
| beta | Float | Market volatility measure |
| sharpe_ratio | Float | Risk-adjusted return |
| sortino_ratio | Float | Downside risk-adjusted return |
| std_dev_ann_pct | Float | Standard deviation of returns |
| max_drawdown_pct | Float | Maximum loss from peak |
| aum_crore | Float | Assets under management |
| expense_ratio_pct | Float | Fund expense ratio |
| morningstar_rating | Integer | Fund rating (1–5) |
| risk_grade | Text | Risk classification |

---

## 4. fact_nav

NAV time-series data for each mutual fund scheme.

---

## 5. fact_transactions

Investor transaction-level data including SIP, Lumpsum, and Redemption behavior.

---

## 6. fact_performance

Fund return and risk metrics used for performance evaluation.

---

## 📊 Project Summary

This project analyzes mutual fund data using Python, SQL, and data visualization techniques to generate insights on investor behavior, fund performance, and market trends.
"""

with open("docs/data_dictionary.md", "w", encoding="utf-8") as f:
    f.write(data_dictionary)

print("Data dictionary created successfully!")

Data dictionary created successfully!


In [71]:
import os
print(os.listdir("docs"))

['data_dictionary.md']


In [73]:
nav_df['date'] = pd.to_datetime(nav_df['date'])
nav_df = nav_df.sort_values(['amfi_code', 'date'])
nav_df = nav_df.drop_duplicates()
nav_df['nav'] = nav_df.groupby('amfi_code')['nav'].ffill()
nav_df = nav_df[nav_df['nav'] > 0]

In [75]:
nav_df.to_csv("data/processed/nav_history_clean.csv", index=False)

In [77]:
txn_df['transaction_date'] = pd.to_datetime(txn_df['transaction_date'])

txn_df['transaction_type'] = txn_df['transaction_type'].str.strip().str.title()

txn_df = txn_df[txn_df['amount_inr'] > 0]

allowed = ['Sip', 'Lumpsum', 'Redemption']
txn_df = txn_df[txn_df['transaction_type'].isin(allowed)]

allowed_kyc = ['Verified', 'Pending', 'Rejected']
txn_df = txn_df[txn_df['kyc_status'].isin(allowed_kyc)]

In [79]:
txn_df.to_csv("data/processed/investor_transactions_clean.csv", index=False)

In [81]:
num_cols = [
    'return_1yr_pct','return_3yr_pct','return_5yr_pct',
    'benchmark_3yr_pct','expense_ratio_pct'
]

for col in num_cols:
    perf_df[col] = pd.to_numeric(perf_df[col], errors='coerce')

perf_df = perf_df.dropna(subset=num_cols)

perf_df = perf_df[
    (perf_df['expense_ratio_pct'] >= 0.1) &
    (perf_df['expense_ratio_pct'] <= 2.5)
]

In [83]:
perf_df.to_csv("data/processed/scheme_performance_clean.csv", index=False)

In [85]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///bluestock_mf.db")

In [87]:
nav_df.to_sql("fact_nav", engine, if_exists="replace", index=False)
txn_df.to_sql("fact_transactions", engine, if_exists="replace", index=False)
perf_df.to_sql("fact_performance", engine, if_exists="replace", index=False)

40

In [89]:
import pandas as pd

print(len(nav_df))
print(pd.read_sql("SELECT COUNT(*) FROM fact_nav", engine))

46000
   COUNT(*)
0     46000


In [91]:
pd.read_sql("SELECT amfi_code, SUM(nav) FROM fact_nav GROUP BY amfi_code ORDER BY SUM(nav) DESC LIMIT 5", engine)

,amfi_code,SUM(nav)
0,120844,4.261758e+06
1,125497,9.170726e+05
2,100016,6.511199e+05
3,149322,5.050531e+05
4,101206,5.013718e+05


In [93]:
with open("docs/data_dictionary.md","w") as f:
    f.write("your documentation text")

In [95]:
git add .
git commit -m "Day 2: Cleaned data + SQLite DB loaded"
git push origin main

SyntaxError: invalid syntax (4154700159.py, line 1)